# Cyclistic Bike Share Analysis

## Notebook 3 - Feature Engineering

Objective:
Create analysis-ready features for Exploratory Data Analysis (EDA).

Features to be added:
- "ride_date" - for daily trend analysis
- "ride_month" and "ride_month_num" (numeric) - for monthly trend analysis
- "day_of_week" - for weekday analysis
- "start_hour" - for hourly usage analysis
- "is_weekend" - for weekend vs weekday
- "time_period" - morning/afternoon/evening

Input: 00_Data/01_Interim/cyclistic_202506_202605_cleaned.parquet

Output: 00_Data/02_Processed/cyclistic_202506_202605_featured.parquet

### Import Libraries and Dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_parquet(
    "../00_Data/01_Interim/cyclistic_202506_202605_cleaned.parquet"
)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5842803 entries, 0 to 5842802
Data columns (total 15 columns):
 #   Column              Dtype          
---  ------              -----          
 0   ride_id             object         
 1   rideable_type       object         
 2   started_at          datetime64[ns] 
 3   ended_at            datetime64[ns] 
 4   start_station_name  object         
 5   start_station_id    object         
 6   end_station_name    object         
 7   end_station_id      object         
 8   start_lat           float64        
 9   start_lng           float64        
 10  end_lat             float64        
 11  end_lng             float64        
 12  member_casual       object         
 13  ride_duration       timedelta64[ns]
 14  ride_duration_min   float64        
dtypes: datetime64[ns](2), float64(5), object(7), timedelta64[ns](1)
memory usage: 668.7+ MB


### Create column "ride_date"

In [4]:
# Define "ride_date" based on trip start date

df["ride_date"] = df["started_at"].dt.date

In [5]:
# Validation

df["ride_date"].head()

0    2025-06-05
1    2025-06-14
2    2025-06-09
3    2025-06-15
4    2025-06-19
Name: ride_date, dtype: object

In [6]:
df["ride_date"].isna().sum()

np.int64(0)

In [7]:
df["ride_date"].nunique()

366

### Create column "ride_month" and "ride_month_num"

In [8]:
# Define "ride_month" based on trip start date

df["ride_month"] = df["started_at"].dt.month_name()

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

df["ride_month"] = pd.Categorical(
    df["ride_month"],
    categories=month_order,
    ordered=True
)

In [9]:
# Validation

df["ride_month"].dtype

CategoricalDtype(categories=['January', 'February', 'March', 'April', 'May', 'June',
                  'July', 'August', 'September', 'October', 'November',
                  'December'],
, ordered=True, categories_dtype=object)

In [10]:
df["ride_month"].value_counts(sort=False)

ride_month
January      137589
February     201339
March        316752
April        447898
May          653260
June         677847
July         762470
August       789601
September    713974
October      645490
November     356163
December     140420
Name: count, dtype: int64

In [11]:
df["ride_month"].isna().sum()

np.int64(0)

In [12]:
# Create an extra column "ride_month_num" - numeric data for future use

df["ride_month_num"] = df["started_at"].dt.month

In [13]:
# Validation

df["ride_month_num"].dtype

dtype('int32')

In [14]:
df["ride_month_num"].isna().sum()

np.int64(0)

In [15]:
df["ride_month_num"].value_counts().sort_index()

ride_month_num
1     137589
2     201339
3     316752
4     447898
5     653260
6     677847
7     762470
8     789601
9     713974
10    645490
11    356163
12    140420
Name: count, dtype: int64

### Create column "day_of_week"

In [16]:
# Define "day_of_week" based on trip start day

df["day_of_week"] = df["started_at"].dt.day_name()

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

df["day_of_week"] = pd.Categorical(
    df["day_of_week"],
    categories=weekday_order,
    ordered=True
)

In [17]:
# Validation

df["day_of_week"].dtype

CategoricalDtype(categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
                  'Saturday', 'Sunday'],
, ordered=True, categories_dtype=object)

In [18]:
df["day_of_week"].value_counts(sort=False)

day_of_week
Monday       771832
Tuesday      826785
Wednesday    830329
Thursday     867891
Friday       871193
Saturday     901149
Sunday       773624
Name: count, dtype: int64

In [19]:
df["day_of_week"].isna().sum()

np.int64(0)

### Create column "start_hour"

In [20]:
# Define "start_hour" based on trip start hour
# Hour range:
# 0-23

df["start_hour"] = df["started_at"].dt.hour

In [21]:
# Validation

df["start_hour"].describe().round(0)

count    5842803.0
mean          14.0
std            5.0
min            0.0
25%           10.0
50%           15.0
75%           18.0
max           23.0
Name: start_hour, dtype: float64

In [22]:
df["start_hour"].value_counts().sort_index()

start_hour
0      75332
1      47586
2      30323
3      18030
4      16973
5      48354
6     133453
7     261557
8     347009
9     247019
10    237009
11    286588
12    331772
13    332807
14    343902
15    411628
16    540622
17    605490
18    484911
19    346484
20    245546
21    197463
22    152376
23    100569
Name: count, dtype: int64

In [23]:
df["start_hour"].isna().sum()

np.int64(0)

### Create column "is_weekend"

In [ ]:
# Define if the trip started on weekend
# Boolean type:
# Weekend = Ture
# Weekday = False

df["is_weekend"] = (
    df["started_at"].dt.weekday >= 5
)

In [25]:
# Validation

df["is_weekend"].value_counts()

is_weekend
False    4168030
True     1674773
Name: count, dtype: int64

In [26]:
df["is_weekend"].isna().sum()

np.int64(0)

### Create column "time_period"

In [27]:
# Define time period based on trip start hour
# Time period definition:
# 00:00-05:59 -> Night
# 06:00-11:59 -> Morning
# 12:00-17:59 -> Afternooon
# 18:00-23:59 -> Evening

bins = [0, 6, 12, 18, 24]

labels = [
    "00-05 (Night)",
    "06-11 (Morning)",
    "12-17 (Afternoon)",
    "18-23 (Evening)"
]

df["time_period"] = pd.cut(
    df["start_hour"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

In [28]:
# Validation

df["time_period"].value_counts(sort=False)

time_period
00-05 (Night)         236598
06-11 (Morning)      1512635
12-17 (Afternoon)    2566221
18-23 (Evening)      1527349
Name: count, dtype: int64

In [29]:
df["time_period"].dtype

CategoricalDtype(categories=['00-05 (Night)', '06-11 (Morning)', '12-17 (Afternoon)',
                  '18-23 (Evening)'],
, ordered=True, categories_dtype=object)

In [30]:
df["time_period"].isna().sum()

np.int64(0)

### Validation and Export

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5842803 entries, 0 to 5842802
Data columns (total 22 columns):
 #   Column              Dtype          
---  ------              -----          
 0   ride_id             object         
 1   rideable_type       object         
 2   started_at          datetime64[ns] 
 3   ended_at            datetime64[ns] 
 4   start_station_name  object         
 5   start_station_id    object         
 6   end_station_name    object         
 7   end_station_id      object         
 8   start_lat           float64        
 9   start_lng           float64        
 10  end_lat             float64        
 11  end_lng             float64        
 12  member_casual       object         
 13  ride_duration       timedelta64[ns]
 14  ride_duration_min   float64        
 15  ride_date           object         
 16  ride_month          category       
 17  ride_month_num      int32          
 18  day_of_week         category       
 19  start_hour          i

In [32]:
df.shape

(5842803, 22)

In [38]:
OUTPUT_PATH = Path("../00_Data/02_Processed")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

assert df["ride_duration_min"].min() >= 0
assert df["ride_duration_min"].max() < 1440

assert df["ride_month_num"].between(1,12).all()

assert df["start_hour"].between(0,23).all()

assert set(df["is_weekend"].unique()) == {True, False}

expected = {
    "00-05 (Night)",
    "06-11 (Morning)",
    "12-17 (Afternoon)",
    "18-23 (Evening)"
}
assert set(df["time_period"].unique()) == expected

new_col = [
    "ride_date",
    "ride_month",
    "ride_month_num",
    "day_of_week",
    "start_hour",
    "is_weekend",
    "time_period"]

assert df[new_col].isna().sum().sum() == 0

assert list(df["day_of_week"].cat.categories) == [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

assert list(df["ride_month"].cat.categories) == [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

In [41]:
df.to_parquet(
    OUTPUT_PATH / "cyclistic_202506_202605_featured.parquet",
    index=False
)

In [43]:
# Export validation

test = pd.read_parquet(
    OUTPUT_PATH / "cyclistic_202506_202605_featured.parquet"
)

assert test.shape == df.shape

assert list(test.columns) == list(df.columns)

In [46]:
print(f"""
Feature Engineering Completed

Rows    : {len(df):,}
Columns : {df.shape[1]}

Output:
{OUTPUT_PATH}
""")


Feature Engineering Completed

Rows    : 5,842,803
Columns : 22

Output:
..\00_Data\02_Processed

